# MMER-XGBoost parametric bootstrap

This notebook uses a parametric bootstrap to characterize the sampling variation of MMER-XGBoost random-effect covariance estimates and derived PC summaries. Outcomes are simulated from a free-living-meal point fit while participant designs and scaling constants are held fixed; each simulated dataset is refitted.

## Notebook flow

1. Load the prepared data and define the fixed-design simulation and refitting functions.
2. Fit the reference MMER-XGBoost model and calculate its covariance eigenspectrum and participant BLUP scores.
3. Run 1,000 parametric bootstrap refits and save covariance matrices, fit status, and simulated participant BLUPs.
4. Summarize full-covariance variance shares, effective dimensionality, aligned loadings, and participant PC-score distributions.
5. Repeat the covariance analysis for macronutrient slopes conditional on random intercepts using the Schur complement.

## Outputs

CSV files written to `code/bootstraps/`:

| File | Contents |
| --- | --- |
| `total_covariance/parametric_bootstraps_variance_explained.csv` | Full-covariance PC variance shares, percentile intervals, and axis-alignment diagnostics. |
| `total_covariance/parametric_bootstraps_effective_dim.csv` | Full-covariance effective dimensionality and percentile intervals. |
| `total_covariance/parametric_bootstraps_loadings.csv` | Aligned full-covariance PC loadings and percentile intervals. |
| `PC_scores_bootstraps.csv` | Participant PC1–PC3 point scores and simulated bootstrap distributions. |
| `conditionned_macronutrient_slopes_covariance/parametric_macro_conditioned_variance_explained.csv` | Conditional macronutrient-slope PC variance shares and percentile intervals. |
| `conditionned_macronutrient_slopes_covariance/parametric_macro_conditioned_effective_dim.csv` | Conditional covariance effective dimensionality and percentile intervals. |
| `conditionned_macronutrient_slopes_covariance/parametric_macro_conditioned_loadings.csv` | Conditional PC loadings and percentile intervals. |
| `conditionned_macronutrient_slopes_covariance/parametric_macro_conditioned_conditioning_variance.csv` | Variance retained and removed by conditioning on intercepts. |
| `conditionned_macronutrient_slopes_covariance/parametric_macro_conditioned_conditioning_condnum.csv` | Condition-number diagnostics for the intercept covariance block. |

The same directory also receives `parametric_bootstrap_geometry_matrices.npz`, `G_point.npy`, and `subject_levels.npy`. The participant-score CSV and subject-level file are Git-ignored.

## 1. Setup

Resolve project paths, import the shared modeling utilities, and load the prepared meal-level dataset.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

# Find the project when launched from its root, code/, or a notebook subfolder.
for PROJECT_ROOT in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    CODE_DIR = PROJECT_ROOT / "code"
    if (CODE_DIR / "data_paths.py").is_file():
        break
else:
    raise FileNotFoundError("Open this notebook from within the project directory.")

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# Shared defaults; override individual paths here if needed.
from data_paths import DATA_DIR, METADATA_PATH, MEAL_DATA_PATH, CGM_METRICS_PATH
FIGURES_DIR = PROJECT_ROOT / "Figures"
RESULTS_DIR = PROJECT_ROOT / "Results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

from utils import build_design_mats, load_and_prepare_data, make_xgb

In [2]:
meta_data, data, id_to_subject_key, clusters = load_and_prepare_data(
    METADATA_PATH,
    MEAL_DATA_PATH,
)

print(
    f"Prepared {len(data):,} meals from "
    f"{data['subject_key'].nunique():,} participants."
)

data shape : (54987, 133)
Prepared 54,987 meals from 992 participants.


In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr
from tqdm import tqdm
import statsmodels.api as sm
import logging
import statsmodels.formula.api as smf

from mmer import MixedEffectEstimator

## 2. Parametric bootstrap design

Simulate random effects and observation errors from the point-fit covariance estimates while retaining the observed participant designs and scaling constants. Refit MMER-XGBoost to each simulated outcome matrix.

### Simulation and refitting functions

In [4]:
# Parametric bootstrap for covariance geometry in scaled MMER.
# Simulate outcomes from the fitted model while retaining all participants
# and their within-person designs. Simulate and refit in the original fit's
# z-scored outcome space, using fixed design matrices and scaling constants.
# Per replicate (held fixed: X, groups, f_hat(X), G_hat, R_hat):
#   u_g* ~ N(0, G_hat)   for each distinct subject      (20-dim)
#   e_i* ~ N(0, R_hat)   for each observation           (4-dim)
#   y_i* = f_hat(X_i) + [1, X_i[0:4]] . u*_{subj(i)} + e_i*
#   refit XGBoost + EM on (X, y*, groups)  ->  store G_b*, R_b*
# Output: parametric_bootstrap_geometry_matrices.npz, containing replicate
# identifiers, fit status, covariance matrices, and participant BLUPs.

import os, sys, contextlib
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from joblib import Parallel, delayed
from mmer import MixedEffectEstimator

OUTDIR = CODE_DIR / "bootstraps"
os.makedirs(OUTDIR, exist_ok=True)

prediction_metrics = ["max_glucose", "peak_duration", "end_glucose", "positive_iAUC"]
slope_variables    = ["carb_eaten", "fat_eaten", "protein_eaten", "fiber_eaten"]
RANDOM_SLOPE_COLS  = [0, 1, 2, 3]                 # X cols used as random-slope covariates
N_OUTCOMES, N_EFFECTS = 4, 5
MATRIX_DIM = N_OUTCOMES * N_EFFECTS               # 20


@contextlib.contextmanager
def suppress_stdout_stderr():
    with open(os.devnull, "w") as devnull:
        old = sys.stdout, sys.stderr
        try:
            sys.stdout, sys.stderr = devnull, devnull
            yield
        finally:
            sys.stdout, sys.stderr = old


def symmetrize(A):
    A = np.asarray(A, float)
    return 0.5 * (A + A.T)


def _psd_factor(A, jitter=1e-12):
    """F with F @ F.T ~= A (PSD-projected). Samples: z @ F.T, z ~ N(0, I)."""
    A = symmetrize(A)
    w, V = np.linalg.eigh(A)
    w = np.clip(w, 0, None)
    return V * np.sqrt(w + jitter)


# Use the shared fixed-effect model from utils.py.
def _build_estimator(seed, xgb_n_jobs, mmer_n_jobs):
    xgb_fe = make_xgb(njobs=xgb_n_jobs)
    xgb_fe.set_params(random_state=seed, verbosity=0)
    return MixedEffectEstimator(xgb_fe, max_iter=50, slq_steps=40, tol=1e-7, n_jobs=mmer_n_jobs)


def build_design_once(data, fixed_scalers):
    """Build X_df, scaled outcomes and groups with the point-fit configuration."""
    with suppress_stdout_stderr():
        (X_train, _, Z_train, _, y_train, _, clusters_train, _, cluster_mapping,
         info_, _, validation_dataframe) = build_design_mats(
            data, full_train=True, target_outcome=prediction_metrics,
            apply_mundlak=False, use_interactions=[False],
            population_z_scoring_features=True, remove_std_meals_from_training=True,
            remove_features=["demographics"], random_slopes=slope_variables,
            population_z_scoring_outcome=True, fixed_scalers=fixed_scalers,
        )
    y = np.asarray(y_train)
    if y.ndim == 1:
        y = y.reshape(-1, 1)
    groups = clusters_train.to_numpy().reshape(-1, 1)
    return X_train, y, groups        # keep X_train as DataFrame (categoricals intact)


def fit_point_model(X_df, y, groups, seed=42, xgb_n_jobs=4, mmer_n_jobs=4):
    with suppress_stdout_stderr():
        est = _build_estimator(seed, xgb_n_jobs, mmer_n_jobs)
        return est.fit(X_df, y, groups, random_slopes=(RANDOM_SLOPE_COLS,))

# simulate one synthetic outcome matrix (in the scaled space)
def simulate_outcomes(mu, Z, inv, n_sub, F_G, F_R, rng):
    """mu:(N,m) FE mean ; Z:(N,q)=[1, X[:,0:4]] ; inv:(N,) subject idx ; F_*: PSD factors."""
    N, m = mu.shape
    q = N_EFFECTS
    U = (rng.standard_normal((n_sub, m * q)) @ F_G.T).reshape(n_sub, m, q)
    re = np.einsum("nq,noq->no", Z, U[inv])            # (N, m) random part
    E = rng.standard_normal((N, m)) @ F_R.T            # (N, m) residuals
    return mu + re + E


def one_parametric_refit(b, X_df, Z, inv, n_sub, mu, F_G, F_R, seed, xgb_n_jobs, mmer_n_jobs):
    rng = np.random.default_rng(seed + b)
    groups = inv.reshape(-1, 1)
    try:
        y_star = simulate_outcomes(mu, Z, inv, n_sub, F_G, F_R, rng)
        with suppress_stdout_stderr():
            est = _build_estimator(seed + b, xgb_n_jobs, mmer_n_jobs)
            fit = est.fit(X_df, y_star, groups, random_slopes=(RANDOM_SLOPE_COLS,))

        # Align participant BLUPs by group code (0..n_sub-1) across replicates.
        blup_b = fit.blups(X=X_df, y=y_star, groups=groups, group_idx=0)
        blup_b = blup_b.reindex(range(n_sub))     # guard against any dropped codes
        B_b = blup_b.to_numpy(dtype=np.float64)   # (n_sub, D)

        return dict(bootstrap=b, status="success",
                    G=symmetrize(fit.G[0]), R=symmetrize(fit.R),
                    B=B_b,
                    converged=getattr(fit, "is_converged", np.nan),
                    ll=getattr(fit, "best_log_likelihood", np.nan))
    except Exception as e:
        return dict(bootstrap=b, status="failed", error=repr(e), G=None, R=None, B=None)

In [5]:
def run_parametric_bootstrap(
    X_df, y, groups, point_model, B=1000,
    n_parallel_jobs=10, xgb_n_jobs=4, mmer_n_jobs=None, seed=42,
    out_prefix="parametric_bootstrap_geometry",
):
    """
    Run a parametric bootstrap using a fitted model and its design matrices.
    The point estimate, design and scaling constants remain fixed.

    Example:

        run_parametric_bootstrap(
            X_train, y_train_mmer, groups_train_mmer,
            point_model=final_model_result,
            B=1000, n_parallel_jobs=10, xgb_n_jobs=4, seed=42,
        )
    n_parallel_jobs : outer joblib processes (independent refits at once)
    xgb_n_jobs      : threads inside XGBoost per bootstrap refit
    mmer_n_jobs     : threads inside the EM per bootstrap refit (defaults to xgb_n_jobs)
    Keep n_parallel_jobs * xgb_n_jobs <= physical cores to avoid oversubscription.
    """
    if mmer_n_jobs is None:
        mmer_n_jobs = xgb_n_jobs

    y = np.asarray(y)
    if y.ndim == 1:
        y = y.reshape(-1, 1)
    groups = np.asarray(groups).reshape(-1, 1)

    G_hat = symmetrize(point_model.G[0])
    R_hat = symmetrize(point_model.R)
    F_G, F_R = _psd_factor(G_hat), _psd_factor(R_hat)

    mu = np.asarray(point_model.predict(X_df))                    # FE mean (scaled space)
    if mu.ndim == 1:
        mu = mu.reshape(-1, 1)

    if isinstance(X_df, pd.DataFrame):
        Z_cov = X_df.iloc[:, RANDOM_SLOPE_COLS].to_numpy(dtype=float)
    else:
        Z_cov = np.asarray(X_df)[:, RANDOM_SLOPE_COLS].astype(float)
    Z = np.column_stack([np.ones(len(mu)), Z_cov])                # (N, q)

    levels, inv = np.unique(groups.ravel(), return_inverse=True)
    n_sub = len(levels)
    inv = inv.astype(np.int64)

    print("=" * 70)
    print("Parallel parametric bootstrap (reusing supplied point model)")
    print(f"B={B}  n_subjects={n_sub}  n_obs={len(mu)}")
    print(f"outer jobs={n_parallel_jobs}  xgb threads={xgb_n_jobs}  emm threads={mmer_n_jobs}"
          f"  (~{n_parallel_jobs * xgb_n_jobs} cores)")
    print("=" * 70)

    tasks = [delayed(one_parametric_refit)(
        b, X_df, Z, inv, n_sub, mu, F_G, F_R, seed, xgb_n_jobs, mmer_n_jobs
    ) for b in range(B)]

    try:
        gen = Parallel(n_jobs=n_parallel_jobs, backend="loky",
                       return_as="generator_unordered")(tasks)
    except (TypeError, ValueError):
        gen = Parallel(n_jobs=n_parallel_jobs, backend="loky",
                       return_as="generator")(tasks)
    rows = list(tqdm(gen, total=B, desc="Parametric refits", unit="fit"))

    # Keep failed replicates as NaN arrays to preserve replicate alignment.
    nan_G = np.full((MATRIX_DIM, MATRIX_DIM), np.nan)
    nan_R = np.full((N_OUTCOMES, N_OUTCOMES), np.nan)
    nan_B = np.full((n_sub, MATRIX_DIM), np.nan)

    order = np.argsort([r["bootstrap"] for r in rows])
    rows = [rows[i] for i in order]
    G_stack = [r.get("G") if r.get("G") is not None else nan_G for r in rows]
    R_stack = [r.get("R") if r.get("R") is not None else nan_R for r in rows]
    B_stack = [r.get("B") if r.get("B") is not None else nan_B for r in rows]

    npz_path = os.path.join(OUTDIR, f"{out_prefix}_matrices.npz")
    np.savez_compressed(
        npz_path,
        bootstrap=np.array([r["bootstrap"] for r in rows]),
        status=np.array([r["status"] for r in rows]),
        G_full=np.stack(G_stack), R=np.stack(R_stack),
        B_full=np.stack(B_stack).astype(np.float32),      # (B, n_sub, D); float32 limits file size.

    )
    np.save(os.path.join(OUTDIR, "G_point.npy"), G_hat)
    np.save(os.path.join(OUTDIR, "subject_levels.npy"), levels)


    n_ok = sum(r["status"] == "success" for r in rows)
    print(f"\nsuccessful refits: {n_ok}/{B}")
    print(f"[saved] {npz_path}")
    print(f"[saved] {os.path.join(OUTDIR, 'G_point.npy')}")

    # Estimate bootstrap bias at the observed sample size.
    Gb = np.stack([g for g, r in zip(G_stack, rows) if r["status"] == "success"])
    rel = np.where(np.abs(np.diag(G_hat)) > 1e-12,
                   (np.diag(Gb.mean(0)) - np.diag(G_hat)) / np.abs(np.diag(G_hat)), np.nan)
    diag = pd.DataFrame({
        "term": [f"{o}::{e}" for o in prediction_metrics for e in ["intercept", *slope_variables]],
        "point": np.diag(G_hat), "boot_mean": np.diag(Gb.mean(0)), "rel_bias": rel})
    pd.set_option("display.float_format", lambda v: f"{v:9.4f}")
    print("\n== estimator bias at true N (parametric) ==")
    print(diag.to_string(index=False))

    return point_model, npz_path

## 3. Point estimate

Fit the reference model to free-living meals and calculate its random-effect covariance and participant PC scores.

In [6]:
##################################### random intercept + random slopes
# Common cleaning and eligibility filters were applied by load_and_prepare_data().
data = data.copy().reset_index(drop=True)

groups = data["subject_key"].astype('category').cat.codes.values
clusters = pd.Series(groups)  # subject codes



results_list = []

prediction_metrics = ["max_glucose", "peak_duration", "end_glucose", "positive_iAUC",
                     ]
print("Dataset size:", data.shape)

print("Build the design matrix....")
X_train, _, Z_train, _, y_train, _, clusters_train, _, cluster_mapping, info_, _, validation_dataframe = \
    build_design_mats(
        data,
        full_train=True,
        target_outcome=prediction_metrics,
        apply_mundlak=False,
        use_interactions=[False],
        population_z_scoring_features=True,
        remove_std_meals_from_training=True,
        remove_features=["demographics"],
        random_slopes=["carb_eaten", "fat_eaten", "protein_eaten", "fiber_eaten"],
        population_z_scoring_outcome=True,
    )

print("Train the model...")
xgb_fe = make_xgb(njobs=10)

point_estimate_model = MixedEffectEstimator(xgb_fe, max_iter=50, slq_steps=40, tol=1e-07, n_jobs=8)

groups_train_mmer = clusters_train.to_numpy().reshape(-1, 1)
y_train_mmer = np.asarray(y_train)
if y_train_mmer.ndim == 1:
    y_train_mmer = y_train_mmer.reshape(-1, 1)

pe_model = point_estimate_model.fit(X_train, y_train_mmer, groups_train_mmer,random_slopes=([0, 1, 2, 3],))


# Scaling constants for unscaling and held-out validation.
_, _, _, _, _, _, _, _, _, info_ref, _, _ = \
    build_design_mats(
        data, full_train=True, target_outcome=prediction_metrics,
        apply_mundlak=False, use_interactions=[False],
        population_z_scoring_features=True, remove_std_meals_from_training=True,
        remove_features=["demographics"],
        random_slopes=["carb_eaten", "fat_eaten", "protein_eaten", "fiber_eaten"],
        population_z_scoring_outcome=True,
    )

fixed_scalers = {
    "z_score_mean": info_ref["z_score_mean"],
    "z_score_std": info_ref["z_score_std"],
    "outcome_mean": info_ref["outcome_mean"],
    "outcome_std": info_ref["outcome_std"]}


if "other_mean" in info_ref:
    fixed_scalers["other_mean"] = info_ref["other_mean"]
    fixed_scalers["other_std"] = info_ref["other_std"]

Dataset size: (54987, 133)
Build the design matrix....
Train the model...


Finished: no further improvement!:  66%|██████▌   | 33/50 08:21         


In [8]:
def covariance_pca(G, labels=None, blups=None, cluster_mapping=None, clip_negative=True):
    G = np.asarray(G, float)
    G = 0.5 * (G + G.T)  # symmetrize

    eigvals, eigvecs = np.linalg.eigh(G)
    idx = np.argsort(eigvals)[::-1]
    eigvals = eigvals[idx]
    eigvecs = eigvecs[:, idx]

    var = np.clip(eigvals, 0, None) if clip_negative else eigvals
    variance_explained = pd.Series(
        var / var.sum(),
        index=[f"PC{i+1}" for i in range(len(eigvals))],
        name="variance_explained",
    )

    if labels is None:
        labels = [f"V{i+1}" for i in range(G.shape[0])]

    loadings = pd.DataFrame(eigvecs, index=labels, columns=variance_explained.index)

    print("Variance explained:")
    print((100 * variance_explained[:4]).round(2).astype(str) + "%")

    scores = None
    if blups is not None:
        X = blups.values if isinstance(blups, pd.DataFrame) else np.asarray(blups)
        X = X - X.mean(axis=0)
        scores = pd.DataFrame(
            X @ eigvecs,
            index=blups.index if isinstance(blups, pd.DataFrame) else None,
            columns=variance_explained.index,
        )
        if cluster_mapping is not None:
            scores = scores.reset_index().rename(columns={"Group_0": "cluster"})
            scores = scores.merge(cluster_mapping, on="cluster", how="inner")

    return loadings, variance_explained, scores
    
# Get BLUPs
blups = pe_model.blups(
    X=X_train,
    y=y_train_mmer,
    groups=groups_train_mmer,
    group_idx=0)

effect_names = ["Intercept", "Carb", "Fat", "Protein", "Fiber"]
outcome_names = ["max_glucose", "peak_duration", "end_glucose", "positive_iAUC",]
labels = [f"{y}|{e}" for y in outcome_names for e in effect_names]

blups.columns = labels
G_full = pe_model.G[0]
# PCA
loadings, var_exp, scores = covariance_pca(
    G=pe_model.G[0],
    labels=labels,
    blups=blups,
    cluster_mapping = cluster_mapping
)

Variance explained:
PC1    69.32%
PC2    11.79%
PC3     5.04%
PC4     4.66%
Name: variance_explained, dtype: object


## 4. Parametric bootstrap refits

In [9]:
# Bootstrap the fitted model with its original design matrices.
point_model, npz = run_parametric_bootstrap(
    X_train, y_train_mmer, groups_train_mmer, point_model=pe_model,
    B=1000, n_parallel_jobs=16, xgb_n_jobs=4, seed=42,)

Parallel parametric bootstrap (reusing supplied point model)
B=1000  n_subjects=992  n_obs=50463
outer jobs=16  xgb threads=4  emm threads=4  (~64 cores)


Parametric refits:   0%|          | 0/1000 [00:00<?, ?fit/s]


successful refits: 1000/1000
[saved] code/bootstraps/parametric_bootstrap_geometry_matrices.npz
[saved] code/bootstraps/G_point.npy

== estimator bias at true N (parametric) ==
                        term     point  boot_mean  rel_bias
      max_glucose::intercept    0.0956     0.0955   -0.0011
     max_glucose::carb_eaten    0.0386     0.0388    0.0057
      max_glucose::fat_eaten    0.0055     0.0062    0.1261
  max_glucose::protein_eaten    0.0073     0.0075    0.0382
    max_glucose::fiber_eaten    0.0069     0.0072    0.0453
    peak_duration::intercept    0.0444     0.0448    0.0101
   peak_duration::carb_eaten    0.0043     0.0064    0.4744
    peak_duration::fat_eaten    0.0029     0.0051    0.7404
peak_duration::protein_eaten    0.0032     0.0056    0.7688
  peak_duration::fiber_eaten    0.0033     0.0051    0.5591
      end_glucose::intercept    0.1030     0.1027   -0.0033
     end_glucose::carb_eaten    0.0355     0.0367    0.0329
      end_glucose::fat_eaten    0.0063    

## 5. Full random-effect covariance

In [10]:
# Post-processing for parametric bootstrap geometry.
# Inputs (from the parametric runner):
#   parametric_bootstrap_geometry_matrices.npz   (G_full, R, status, bootstrap)
#   G_point.npy                                  (the matching point estimate)
# Compute variance-explained shares from each replicate's eigenvalue spectrum
# and summarize uncertainty with percentile intervals. Measure axis alignment
# separately using cosine similarity to the reference eigenvectors.
# Produces, with point estimate + 95% percentile CI:
#   - PCA variance explained per PC (+ cumulative), from each replicate's own
#     eigenvalue spectrum (rotation-invariant share of total variance)
#   - PC loadings (eigenvectors aligned to the real axes, sign-fixed)
#   - effective dimensionality
#   - an alignment-cosine diagnostic: how stably each PC is recovered
# Covariance blocks supported by run():
#   effects: decompose the selected principal submatrix (marginal covariance).
#   condition_on: decompose the residual covariance after conditioning,
#                 tau_ss - tau_sc tau_cc^{-1} tau_cs (Schur complement).
# All reported percentages are relative to the decomposed block's total
# variance (the residual block's variance when conditioning).
# SIGN CONVENTION (interpretability only; eigenvector signs are arbitrary):
#   Each PC is oriented so that, if a carbohydrate loading is among its top-3
#   loadings by magnitude, the dominant carb loading comes out POSITIVE.
#   Axes with no carb in the top-3 fall back to "largest loading positive" so
#   orientation is deterministic across runs. Controlled by orient_prefer /
#   orient_top on run(); orient_prefer=None disables the convention.

import os
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

OUTCOMES = ["max_glucose", "peak_duration", "end_glucose", "positive_iAUC"]
EFFECTS  = ["intercept", "carb", "fat", "protein", "fiber"]
M, Q = len(OUTCOMES), len(EFFECTS)
D = M * Q                                                  # 20
LABELS = [f"{o}::{e}" for o in OUTCOMES for e in EFFECTS]  # outcome-major, effect-minor


# helpers
def sym(A):
    A = np.asarray(A, float)
    return 0.5 * (A + A.T)

def eig_desc(A):
    w, V = np.linalg.eigh(sym(A))
    return w[::-1], V[:, ::-1]                              # descending

def schur_complement(G, idx_s, idx_c, rcond=1e-10):
    """Residual covariance of block s given block c:
        tau_ss - tau_sc tau_cc^{-1} tau_cs
    Uses a pseudo-inverse (rcond guard) because tau_cc (the conditioning block,
    e.g. the correlated intercepts) can be poorly conditioned. Returns a
    symmetric len(idx_s) x len(idx_s) matrix."""
    Sss = G[np.ix_(idx_s, idx_s)]
    Ssc = G[np.ix_(idx_s, idx_c)]
    Scc = G[np.ix_(idx_c, idx_c)]
    Scc_inv = np.linalg.pinv(sym(Scc), rcond=rcond)
    return sym(Sss - Ssc @ Scc_inv @ Ssc.T)


def orient_axes(V, labels, k, prefer="carb", top=3):
    """Impose a deterministic sign convention on the first k eigenvectors.

    For each PC, if any of its top-`top` loadings (by |value|) is a `prefer`
    effect (label of the form 'outcome::prefer'), flip the axis so that the
    largest-|value| `prefer` loading is POSITIVE. Otherwise flip so the single
    largest-|value| loading is positive (deterministic fallback). Eigenvector
    signs carry no statistical meaning, so this only aids interpretation and
    leaves variance-explained / eigenvalues unchanged. Returns a copy."""
    V = V.copy()
    prefer_pos = [a for a, lab in enumerate(labels) if lab.split("::")[-1] == prefer]
    kk = min(k, V.shape[1])
    for j in range(kk):
        col = V[:, j]
        order = np.argsort(-np.abs(col))          # indices, largest |value| first
        top_idx = order[:top]
        carb_in_top = [a for a in top_idx if a in prefer_pos]
        if carb_in_top:
            pivot = max(carb_in_top, key=lambda a: abs(col[a]))
        else:
            pivot = order[0]                       # fallback: dominant loading
        if col[pivot] < 0:
            V[:, j] = -col
    return V

def psd_project(A):
    w, V = np.linalg.eigh(sym(A))
    return (V * np.clip(w, 0, None)) @ V.T


def load(npz_path, point_path):
    """Return (list of valid G_b, point G)."""
    z = np.load(npz_path, allow_pickle=True)
    G, status = z["G_full"], z["status"]
    finite = np.isfinite(G.reshape(len(G), -1)).all(1)
    ok = (status == "success") & finite
    if "converged" in z:                                   # filter non-converged if stored
        ok &= (z["converged"] == True) | (z["converged"] == 1)
    G_list = [sym(g) for g in G[ok]]
    Gp = sym(np.load(point_path))
    Gp = psd_project(Gp)                 # Match the covariance used to simulate bootstrap outcomes.
    print(f"using {len(G_list)} / {len(G)} replicates")
    return G_list, Gp


def resolve_effects(names):
    """Map a list of names to indices into the full D-length label vector.

    Each name may be:
      - a full label     e.g. "max_glucose::carb"
      - an effect type   e.g. "carb"        (that effect across all outcomes)
      - an outcome       e.g. "max_glucose" (all effects for that outcome)

    Returns (idx, labels) with idx in ascending canonical (outcome-major) order.
    Pass names=None to keep all D effects.
    """
    if names is None:
        return list(range(D)), list(LABELS)
    keep = set()
    for name in names:
        if name in LABELS:
            keep.add(LABELS.index(name))
        elif name in EFFECTS:
            keep.update(a for a, lab in enumerate(LABELS)
                        if lab.split("::")[1] == name)
        elif name in OUTCOMES:
            keep.update(a for a, lab in enumerate(LABELS)
                        if lab.split("::")[0] == name)
        else:
            raise ValueError(
                f"unrecognized name: {name!r}. Expected a full label, one of "
                f"EFFECTS={EFFECTS}, or one of OUTCOMES={OUTCOMES}.")
    if not keep:
        raise ValueError("no random effects selected")
    idx = sorted(keep)
    return idx, [LABELS[a] for a in idx]

def align(Vb, V_ref, k):
    """Match a replicate's top-k eigenvectors to the reference axes; fix sign.
    Returns aligned loadings (D, k) and alignment cosines (k,)."""
    S = np.abs(V_ref[:, :k].T @ Vb)                        # (k, D) similarity
    _, cols = linear_sum_assignment(-S)                    # best assignment
    V = Vb[:, cols[:k]].copy()
    for j in range(k):
        if V_ref[:, j] @ V[:, j] < 0:
            V[:, j] *= -1
    cos = np.abs(np.sum(V * V_ref[:, :k], axis=0))
    return V, cos

# derive per-replicate metrics
def derive(G_list, Gp, k=3, labels=None, orient_prefer="carb", orient_top=3):
    w_ref, V_ref = eig_desc(Gp)
    # interpretability sign convention on the reference axes; replicates inherit
    # it via align(), point loadings via pt_load. Variance/eigenvalues unchanged.
    if orient_prefer is not None and labels is not None:
        V_ref = orient_axes(V_ref, labels, k, prefer=orient_prefer, top=orient_top)
    d_dim = Gp.shape[0]                                     # local: may be a sub-block
    B = len(G_list)

    ve   = np.zeros((B, k))                                # variance share (own eigenvalues; rotation-invariant)
    load = np.zeros((B, d_dim, k))                         # aligned loadings
    cos  = np.zeros((B, k))                                # alignment cosines (axis-recovery stability)
    eff  = np.zeros(B)                                     # effective dimensionality

    for i, G in enumerate(G_list):
        wb, Vb = eig_desc(G)                               # replicate's own spectrum + axes
        ve[i] = 100 * wb[:k] / wb.sum()                    # % variance in top-k dims (matches point's construction)
        load[i], cos[i] = align(Vb, V_ref, k)             # how stably each reference axis is recovered
        ev = np.clip(wb, 0, None)
        eff[i] = ev.sum() ** 2 / np.sum(ev ** 2) if ev.any() else np.nan

    cum = np.cumsum(ve, axis=1)

    # point estimates
    pt_ve  = 100 * w_ref[:k] / w_ref.sum()
    pt_cum = np.cumsum(pt_ve)
    ev_p   = np.clip(w_ref, 0, None)
    pt_eff = ev_p.sum() ** 2 / np.sum(ev_p ** 2)


    return dict(ve=ve, cum=cum, load=load, cos=cos, eff=eff, V_ref=V_ref,
                pt_ve=pt_ve, pt_cum=pt_cum, pt_load=V_ref[:, :k], pt_eff=pt_eff)


def ci(col, point):
    lo, hi = np.percentile(col, [2.5, 97.5])
    return dict(point=point, mean=col.mean(), lo=lo, hi=hi)
from scipy.stats import norm

# run + print + save
def run(npz_path, point_path, k=20, top_loadings=10, effects=None,
        condition_on=None, rcond=1e-10, orient_prefer="carb", orient_top=3,
        save_prefix=None):
    """Decompose tau, a sub-block of it, or a conditioned (residual) block.

    effects : list[str] | None
        Names selecting the block to DECOMPOSE (see resolve_effects for accepted
        forms). None -> full tau, unless condition_on is set (then it defaults to
        every effect NOT in condition_on).
    condition_on : list[str] | None
        Names selecting a block to PARTIAL OUT via the Schur complement before
        decomposing `effects`. Must be disjoint from `effects`. None -> plain
        marginalization (no conditioning).
    rcond : float
        Pseudo-inverse cutoff for tau_cc when conditioning.
    orient_prefer : str | None
        Effect whose dominant loading is forced positive per PC (default "carb").
        None disables the sign convention (arbitrary eigh signs).
    orient_top : int
        Window (top-N loadings by magnitude) in which to look for `orient_prefer`.
    """
    G_list, Gp = load(npz_path, point_path)

    # ---- resolve the block to decompose (s) and, optionally, partial out (c)
    idx_c, labels_c = [], []
    if condition_on is not None:
        idx_c, labels_c = resolve_effects(condition_on)

    if effects is None:
        idx_s = [a for a in range(D) if a not in set(idx_c)]
    else:
        idx_s, _ = resolve_effects(effects)
    idx_s = sorted(set(idx_s))
    labels = [LABELS[a] for a in idx_s]

    if condition_on is not None:
        overlap = set(idx_s) & set(idx_c)
        if overlap:
            raise ValueError(
                "effects and condition_on overlap on "
                f"{[LABELS[a] for a in sorted(overlap)]}; "
                "a block cannot be conditioned on itself.")

    # ---- build the target matrices (point + every replicate) ----
    cond_tab = condnum_tab = None
    if condition_on is not None:
        print(f"CONDITIONING (Schur complement): decompose {len(idx_s)} effects "
              f"given {len(idx_c)} effects")
        print(f"  decompose  : {labels}")
        print(f"  partial out: {labels_c}")

        cond_num = np.linalg.cond(sym(Gp[np.ix_(idx_c, idx_c)]))
        flag = "   [WARN: ill-conditioned; interpret with care]" if cond_num > 1e6 else ""
        print(f"  cond(tau_cc) point = {cond_num:.3g}{flag}")

        # retained-variance fraction = trace(residual) / trace(marginal block)
        pt_retained = (np.trace(schur_complement(Gp, idx_s, idx_c, rcond))
                       / np.trace(Gp[np.ix_(idx_s, idx_s)]))
        retained = np.array([
            np.trace(schur_complement(G, idx_s, idx_c, rcond))
            / np.trace(G[np.ix_(idx_s, idx_s)]) for G in G_list])
        conds = np.array([np.linalg.cond(sym(G[np.ix_(idx_c, idx_c)]))
                          for G in G_list])

        Gp = schur_complement(Gp, idx_s, idx_c, rcond)
        G_list = [schur_complement(G, idx_s, idx_c, rcond) for G in G_list]

        ret_pct = 100 * retained
        cond_tab = pd.DataFrame([
            {"metric": "var_retained_%", **ci(ret_pct, 100 * pt_retained)},
            {"metric": "var_removed_%",  **ci(100 - ret_pct, 100 * (1 - pt_retained))},
        ])
        condnum_tab = pd.DataFrame([{
            "metric": "cond_tau_cc", "point": cond_num,
            "boot_median": float(np.median(conds)), "boot_max": float(conds.max()),
        }])
    else:
        if len(idx_s) < D:
            Gp = Gp[np.ix_(idx_s, idx_s)]
            G_list = [G[np.ix_(idx_s, idx_s)] for G in G_list]
            print(f"decomposing sub-block: {len(idx_s)}/{D} random effects")
            print(f"  -> {labels}")

    if k > len(idx_s):
        print(f"[warn] k={k} > selected dim {len(idx_s)}; using k={len(idx_s)}")
        k = len(idx_s)

    if orient_prefer is not None:
        print(f"sign convention: PCs oriented so the dominant '{orient_prefer}' "
              f"loading (if within top {orient_top}) is positive; "
              f"else largest loading positive")

    d = derive(G_list, Gp, k, labels=labels,
               orient_prefer=orient_prefer, orient_top=orient_top)

    # variance explained (+ cumulative)
    ve_tab = pd.DataFrame([{
        "PC": j + 1,
        **{f"var_{x}": v for x, v in ci(d["ve"][:, j], d["pt_ve"][j]).items()},
        **{f"cum_{x}": v for x, v in ci(d["cum"][:, j], d["pt_cum"][j]).items()},
        "median_align_cos": np.median(d["cos"][:, j]),
    } for j in range(k)])

    # effective dimensionality
    eff_tab = pd.DataFrame([{"metric": "eff_dim", **ci(d["eff"], d["pt_eff"])}])

    # loadings (every entry; print top-N per PC)
    load_tab = pd.DataFrame([{
        "PC": j + 1, "variable": labels[a],
        **ci(d["load"][:, a, j], d["pt_load"][a, j]),
    } for j in range(k) for a in range(len(labels))])

    # ---- print ----
    pd.set_option("display.float_format", lambda v: f"{v:8.3f}")

    if cond_tab is not None:
        print("\n== conditioning diagnostics ==")
        print("  variance of the decomposed block after partialling out the "
              "conditioning block")
        print(cond_tab[["metric", "point", "lo", "hi"]].to_string(index=False))
        print(condnum_tab[["metric", "point", "boot_median", "boot_max"]].to_string(index=False))

    scope = "residual (conditioned)" if condition_on is not None else "selected"
    print(f"\n== PCA variance explained (% of {scope} between-person variance), point [95% CI] ==")
    print(ve_tab[["PC", "var_point", "var_lo", "var_hi",
                  "cum_point", "cum_lo", "cum_hi", "median_align_cos"]].to_string(index=False))

    print("\n== effective dimensionality ==")
    print(eff_tab[["metric", "point", "lo", "hi"]].to_string(index=False))

    print(f"\n== PC loadings — top {top_loadings} |point| per PC, point [95% CI] ==")
    for j in range(k):
        sub = load_tab[load_tab.PC == j + 1]
        sub = sub.reindex(sub.point.abs().sort_values(ascending=False).index).head(top_loadings)
        cos = np.median(d["cos"][:, j])
        print(f"-- PC{j+1}  (median alignment cos = {cos:.2f}) --")
        print(sub[["variable", "point", "lo", "hi"]].to_string(index=False), "\n")

    if save_prefix:
        ve_tab.to_csv(f"{save_prefix}_variance_explained.csv", index=False)
        eff_tab.to_csv(f"{save_prefix}_effective_dim.csv", index=False)
        load_tab.to_csv(f"{save_prefix}_loadings.csv", index=False)
        if cond_tab is not None:
            cond_tab.to_csv(f"{save_prefix}_conditioning_variance.csv", index=False)
            condnum_tab.to_csv(f"{save_prefix}_conditioning_condnum.csv", index=False)
        print(f"[saved] {save_prefix}_*.csv")

    out = dict(variance_explained=ve_tab, effective_dim=eff_tab, loadings=load_tab)
    if cond_tab is not None:
        out.update(conditioning_variance=cond_tab, conditioning_condnum=condnum_tab)
    return out

OUTDIR = CODE_DIR / "bootstraps"
npz = os.path.join(OUTDIR, "parametric_bootstrap_geometry_matrices.npz")
pt  = os.path.join(OUTDIR, "G_point.npy")
# 1) full 20x20  (PCs oriented so dominant carb loading is positive)
run(npz, pt, k=3, save_prefix=os.path.join(OUTDIR, "total_covariance/parametric_bootstraps"))

using 1000 / 1000 replicates
sign convention: PCs oriented so the dominant 'carb' loading (if within top 3) is positive; else largest loading positive

== PCA variance explained (% of selected between-person variance), point [95% CI] ==
 PC  var_point   var_lo   var_hi  cum_point   cum_lo   cum_hi  median_align_cos
  1     69.325   64.142   70.971     69.325   64.142   70.971             0.999
  2     11.793   10.229   13.464     81.118   76.191   82.383             0.992
  3      5.035    4.512    6.310     86.153   82.079   87.386             0.909

== effective dimensionality ==
 metric    point       lo       hi
eff_dim    1.997    1.917    2.295

== PC loadings — top 10 |point| per PC, point [95% CI] ==
-- PC1  (median alignment cos = 1.00) --
                variable    point       lo       hi
positive_iAUC::intercept    0.544    0.531    0.551
  max_glucose::intercept    0.464    0.451    0.475
  end_glucose::intercept    0.438    0.415    0.456
     positive_iAUC::carb    0.328

{'variance_explained':    PC  var_point  var_mean   var_lo   var_hi  cum_point  cum_mean   cum_lo  \
 0   1     69.325    67.914   64.142   70.971     69.325    67.914   64.142   
 1   2     11.793    11.818   10.229   13.464     81.118    79.732   76.191   
 2   3      5.035     5.341    4.512    6.310     86.153    85.073   82.079   
 
     cum_hi  median_align_cos  
 0   70.971             0.999  
 1   82.383             0.992  
 2   87.386             0.909  ,
 'effective_dim':     metric    point     mean       lo       hi
 0  eff_dim    1.997    2.078    1.917    2.295,
 'loadings':     PC                  variable    point     mean       lo       hi
 0    1    max_glucose::intercept    0.464    0.463    0.451    0.475
 1    1         max_glucose::carb    0.254    0.254    0.236    0.273
 2    1          max_glucose::fat   -0.023   -0.028   -0.045   -0.012
 3    1      max_glucose::protein   -0.025   -0.027   -0.045   -0.009
 4    1        max_glucose::fiber   -0.019   -0.020   -

## 6. Participant PC-score summaries

Project the observed and simulated participant BLUPs into matched PC axes and export point scores and bootstrap distribution summaries.

In [11]:
def subject_pc_scores(final_model_result, X_train, y_train_mmer, groups_train_mmer,
                       npz_path, point_path, k=3, group_idx=0, cluster_mapping=cluster_mapping):
    data = np.load(npz_path)
    G_list = data["G_full"]          # (B, D, D)
    B_list = data["B_full"]          # (B, n_sub, D)
    status = data["status"]
    Gp = np.load(point_path)         # G_point.npy

    ok = status == "success"
    G_list, B_list = G_list[ok], B_list[ok]

    w_ref, V_ref = eig_desc(Gp)

    blup_df = final_model_result.blups(X=X_train, y=y_train_mmer,
                                        groups=groups_train_mmer, group_idx=group_idx)
    blup_df.columns = LABELS
    subj = blup_df.index
    B0 = (blup_df - blup_df.mean(axis=0)).values
    pt_scores = B0 @ V_ref[:, :k]

    boot_scores = []
    for G_b, B_b in zip(G_list, B_list):
        if np.isnan(G_b).any() or np.isnan(B_b).any():
            continue
        V_b_aligned, _ = align(eig_desc(G_b)[1], V_ref, k)
        Bc = B_b - np.nanmean(B_b, axis=0)     # center this replicate's own BLUPs
        boot_scores.append(Bc @ V_b_aligned)
    boot_scores = np.stack(boot_scores)         # (B_ok, n_subj, k)

    rows = []
    for i, s in enumerate(subj):
        for j in range(k):
            col = boot_scores[:, i, j]
            lo, hi = np.nanpercentile(col, [2.5, 97.5])
            rows.append({
                "cluster": s, "PC": j + 1,
                "point": pt_scores[i, j],
                "boot_mean": np.nanmean(col), "boot_median": np.nanmedian(col),
                "lo": lo, "hi": hi,
            })
    df = pd.DataFrame(rows)
    df = pd.merge(df, cluster_mapping, on="cluster")
    return df#pd.DataFrame(rows)

In [12]:
scores_df = subject_pc_scores(
    pe_model, X_train, y_train_mmer, groups_train_mmer,
    npz_path=os.path.join(OUTDIR, "parametric_bootstrap_geometry_matrices.npz"),
    point_path=os.path.join(OUTDIR, "G_point.npy"),
    k=3,
)
scores_df.to_csv(os.path.join(OUTDIR,"PC_scores_bootstraps.csv"))

## 7. Macronutrient-slope covariance conditional on intercepts

Use the Schur complement to summarize the residual covariance of macronutrient slopes after conditioning on random intercepts.

In [13]:
# 3) conditioned: slopes GIVEN intercepts (Schur complement) -> discriminant
#    validity. Does carbohydrate responsiveness survive removing baseline tone?
res = run(npz, pt, k=4,
    effects=["carb", "fat", "protein", "fiber"],
    condition_on=["intercept"],
    save_prefix=os.path.join(OUTDIR, "conditionned_macronutrient_slopes_covariance/parametric_macro_conditioned"))

using 1000 / 1000 replicates
CONDITIONING (Schur complement): decompose 16 effects given 4 effects
  decompose  : ['max_glucose::carb', 'max_glucose::fat', 'max_glucose::protein', 'max_glucose::fiber', 'peak_duration::carb', 'peak_duration::fat', 'peak_duration::protein', 'peak_duration::fiber', 'end_glucose::carb', 'end_glucose::fat', 'end_glucose::protein', 'end_glucose::fiber', 'positive_iAUC::carb', 'positive_iAUC::fat', 'positive_iAUC::protein', 'positive_iAUC::fiber']
  partial out: ['max_glucose::intercept', 'peak_duration::intercept', 'end_glucose::intercept', 'positive_iAUC::intercept']
  cond(tau_cc) point = 176
sign convention: PCs oriented so the dominant 'carb' loading (if within top 3) is positive; else largest loading positive

== conditioning diagnostics ==
  variance of the decomposed block after partialling out the conditioning block
        metric    point       lo       hi
var_retained_%   50.443   45.743   57.374
 var_removed_%   49.557   42.626   54.257
     metri